In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from scipy.stats import spearmanr, rankdata
from joblib import Parallel, delayed
import warnings
import gc
import argparse
from tqdm.auto import tqdm
import glob
import matplotlib.pyplot as plt
import networkx as nx
import re
import sys


In [2]:
#Path to TwINFER code repository
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"
path_to_simulation_data = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/median_simulation/figure_2_simulations_6000_cells/"
path_to_save_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_2_data/"
os.makedirs(path_to_save_plot_data, exist_ok = True)


In [3]:
#Common path to data files
path_to_input_data = f"{path_to_code_repo}/simulation_example_input_data/"
# Note: This configuration need not match the exact simulation parameters.
# However, ensure the following requirements are met:
# 1. The connectivity matrix has the same number of genes as the simulations being analyzed
#    (multiple base_configs can be created if needed for analyzing networks with different number of genes)
# 2. The twin simulation duration matches the time specified in base_config
# 3. The number of twin pairs in the simulation equals n_cell (the number of parent cells)

base_config = {
    'n_cells': 6000, #Number of cells before division (number of twin pairs)
    'simulation_time_before_division': 1000, #The time used to run the initial cells before division. User must set this time to ensure the population reaches steady state [hours]
    'twin_simulation_time_after_division': 48, #The time twin cells are simulated after division and measurements are stored in the output[hours]
    'twin_measurement_resolution': 1, #The time between each measurement of twin cells [hours]. For example, if twin_sampling_duration is 12 and twin_measurement_resolution is 1, the final dataframe will contain hourly measurements for 12 hours (0 is birth).
    "path_to_connectivity_matrix": f"{path_to_input_data}/connectivity_matrix_A_to_B.txt", #path to the connectivity matrix specifying the GRN to simulate
    "param_csv": f"{path_to_input_data}/median_parameter.csv", #Path to the parameters for all genes and interaction terms
    "rows_to_use": [[0,1]], #Rows in the parameter's csv file for each gene - the length should be equal to number of genes in the system. Example - [0,0] will mean use row 0 parameters for both gene 1 and 2
    "output_folder": f"{path_to_simulation_data}", #Path to folder to store simulation 
    "log_file": f"{path_to_code_repo}/example_simulation_output/example_log.jsonl", #Path to the log file
    "type": "A_to_B",  # Name of the network used -- will be in the filename
    "number_of_parallel_parameters": 1, #Number of parameters to be run in parallel
    "number_of_cores_per_parameter": 10, #Number of cores to be used per parameter (number_of_parallel_parameters * number_of_cores_per_parameter = number of cores in your computer)
}

#Default settings for when the two samples are measured
t1 = 1
t2 = 20

## Helper functions - imports and definitions

### Imports for helper functions

In [ ]:
# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    read_input_matrix,
    split_and_merge_simulations
)
from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle

)
from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

### Helper and organising functions defined in this notebook

In [18]:
def get_many_random_pair_corr(
    path_to_simulation_file,
    base_config,
    t1,
    t2,
    threshold_gene_gene_corr=0.04,
    check_for_steady_state=True,
    plot_correlation_matrices_as_heatmap=True,
    have_any_output=True,
    random_seed=42,
):
    """
    Compute large-scale random-pair difference correlations across clones and time points
    to generate a null distribution for statistical inference and return the distribution along with statistical details. 
    This works very similar to the generate_random_shuffle function in the infer_with_twinfer pipeline (found in correlation_analysis_functions.py)

    Parameters
    ----------
    path_to_simulation_file : str or list[str]
        Simulation input. Accepted formats:
        - str : a single CSV simulation file.
        - list/tuple with 1 file   : treated as a single CSV (loaded directly).
        - list/tuple with >=2 files: passed to `split_and_merge_simulations()`
                                    to merge clone sets across simulations.
        All files must contain the columns:
        ['clone_id', 'time_step', 'replicate', ...].

    base_config : dict
        Configuration dictionary with required keys:
        - "path_to_connectivity_matrix" : str
        - "param_csv" : str
        - "n_cells" : int
        - "twin_simulation_time_after_division" : int/float
        - "twin_measurement_resolution" : int/float

    t1, t2 : int or float
        Measurement time points.

    threshold_gene_gene_corr : float, optional
        Unused in this function; included for interface consistency.

    check_for_steady_state : bool, optional
        Unused here.

    plot_correlation_matrices_as_heatmap : bool, optional
        Unused here.

    have_any_output : bool, optional
        Unused here.

    random_seed : int, optional
        Seed for reproducible clone shuffling.

    Returns
    -------
    random_stats : dict
        Statistical summary of the shuffled null correlation distribution for
        the pair ("gene_1", "gene_2"):
        - 'all_values'     : flatten of all shuffle correlations
        - 'mean_per_pair'  : per-shuffle mean
        - 'std_per_pair'   : per-shuffle std
        - 'percentile_95'  : 95th percentile of |correlation|
        - 'percentile_100' : max of |correlation|
        - 'global_mean'    : mean over all values
        - 'global_std'     : std over all values
    """

    # --------------------------------------------------
    # Load simulation file(s)
    # --------------------------------------------------
    try:
        if isinstance(path_to_simulation_file, str):
            # Single file
            simulation = pd.read_csv(path_to_simulation_file)

        elif isinstance(path_to_simulation_file, (list, tuple)):
            # List of files
            if len(path_to_simulation_file) == 1:
                # Single file inside list → read directly
                simulation = pd.read_csv(path_to_simulation_file[0])
            elif len(path_to_simulation_file) >= 2:
                # Merge mutliple simulation files
                simulation = split_and_merge_simulations(path_to_simulation_file)
            else:
                raise ValueError("List of simulation files is empty.")

        else:
            raise TypeError("path_to_simulation_file must be a string or list/tuple.")

    except Exception as e:
        raise RuntimeError(f"Error reading simulation file(s): {e}")

    # --------------------------------------------------
    # Load connectivity + parameters
    # --------------------------------------------------
    path_to_connectivity_matrix = base_config["path_to_connectivity_matrix"]
    param_df = pd.read_csv(base_config["param_csv"], index_col=0)

    # Read the interaction matrix and number of genes
    n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
    gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]

    # --------------------------------------------------
    # Basic information & sanity checks
    # --------------------------------------------------
    n_clones_simulation = simulation["clone_id"].nunique()

    time_points_base_config = np.arange(
        0,
        base_config["twin_simulation_time_after_division"] + 
        base_config["twin_measurement_resolution"],
        base_config["twin_measurement_resolution"],
    )

    # --------------------------------------------------
    # Randomly split clones into 1:1:2 partition
    # --------------------------------------------------
    np.random.seed(random_seed)
    clone_ids_shuffled = np.random.permutation(n_clones_simulation)

    n1 = n2 = n_clones_simulation // 4
    t1_clones = clone_ids_shuffled[:n1]
    t2_clones = clone_ids_shuffled[n1:n1+n2]
    across_t_clones = clone_ids_shuffled[n1+n2:]

    # Data subsets for each partition
    t1_twins = simulation[(simulation["clone_id"].isin(t1_clones)) &
                          (simulation["time_step"] == t1)]

    t2_twins = simulation[(simulation["clone_id"].isin(t2_clones)) &
                          (simulation["time_step"] == t2)]

    across_t_twin1 = simulation[(simulation["clone_id"].isin(across_t_clones)) &
                                (simulation["time_step"] == t1) &
                                (simulation["replicate"] == 1)]

    across_t_twin2 = simulation[(simulation["clone_id"].isin(across_t_clones)) &
                                (simulation["time_step"] == t2) &
                                (simulation["replicate"] == 2)]

    # Reset indices before concatenation
    t1_twins = t1_twins.reset_index(drop=True)
    t2_twins = t2_twins.reset_index(drop=True)
    across_t_twin1 = across_t_twin1.reset_index(drop=True)
    across_t_twin2 = across_t_twin2.reset_index(drop=True)

    # Combine all partitions
    all_t1_t2_measurements = pd.concat(
        [t1_twins, t2_twins, across_t_twin1, across_t_twin2],
        ignore_index=True
    )

    # --------------------------------------------------
    # Generate shuffled random-pair correlations
    # --------------------------------------------------
    scrambled_random_corr = generate_random_shuffle(
        all_t1_t2_measurements,
        gene_list,
        n_shuffles=10000,
        random_state=42,
    )

    # Only use ("gene_1", "gene_2") to match expected structure
    all_correlations = scrambled_random_corr[("gene_1", "gene_2")]

    # --------------------------------------------------
    # Compute summary statistics
    # --------------------------------------------------
    random_stats = {
        "all_values": all_correlations.flatten(),
        "mean_per_pair": np.mean(all_correlations, axis=0),
        "std_per_pair": np.std(all_correlations, axis=0),
        "percentile_95": np.percentile(np.abs(all_correlations.flatten()), 95),
        "percentile_100": np.percentile(np.abs(all_correlations.flatten()), 100),
        "global_mean": np.mean(all_correlations),
        "global_std": np.std(all_correlations),
    }

    return random_stats

In [16]:
def process_simulation_through_infer_with_twinfer(path_to_simulation_file, sim_type, rep_id, base_config, t1, t2, gene_names=None):
    """
    Run data from a simulation through TwINFER, extract all correlation matrices, and flatten
    them into gene-pair columns for downstream CSV export.

    Parameters
    ----------
    path_to_simulation_file : str or list
        Path or list of paths to simulation file(s). Can be a single file or a list
        that TwINFER merges when `merge_to_multiple_states=True`.

    sim_type : str
        Label describing the simulation condition (e.g., "A_to_B_low_kon").

    rep_id : int
        Replicate identifier for this simulation run.

    base_config : dict
        Configuration dictionary passed directly to TwINFER.

    t1, t2 : int or float
        Time points used for extracting twin measurements.

    gene_names : list[str], optional
        Names of genes used to label gene-pair matrix columns.
        If None, generic names are created.

    Returns
    -------
    record : dict
        A flattened dictionary containing:
        - simulation metadata
        - gene-gene correlation threshold for (gene_1, gene_2)
        - flattened directional matrix
        - flattened gene-gene correlation matrix
        - flattened random-pair correlation matrix (t2)
        - flattened twin-pair correlation matrices (t1 and t2)
    """

    results = infer_with_twinfer(
        path_to_simulation_file,
        merge_to_multiple_states=True,
        base_config=base_config,
        t1=t1, 
        t2=t2,
        check_for_steady_state=False,
        have_any_output=False, 
        show_scrambled_distribution_gene_correlation=False,
        plot_correlation_matrices_as_heatmap=False, 
        return_gene_corr_thresholds=True, 
        match_sim_details=False,
        seed=101010
    )

    # Base metadata
    record = {
        "sim_type": sim_type,
        "rep_id": rep_id,
        "analysis_key": f"{sim_type}_rep_{rep_id}",
        "gene_gene_threshold": results["gene_corr_thresholds"][("gene_1", "gene_2")]
    }

    def matrix_to_gene_pair_columns(matrix, gene_names, prefix=""):
        """
        Convert a square matrix (N×N) into flat dictionary columns labeled by
        'prefix + gene_i_gene_j'.

        Parameters
        ----------
        matrix : array-like or DataFrame
            Matrix to flatten.

        gene_names : list[str]
            Names for each row/column index.

        prefix : str
            Optional prefix (e.g., 'directional_', 'gene_gene_').

        Returns
        -------
        dict
            Flattened key → value mapping.
        """
        columns = {}
        if matrix is not None:
            matrix_array = np.array(matrix)
            n_genes = len(gene_names) if gene_names else matrix_array.shape[0]

            if gene_names is None:
                gene_names = [f"g{i+1}" for i in range(n_genes)]

            for i in range(matrix_array.shape[0]):
                for j in range(matrix_array.shape[1]):
                    key = f"{prefix}{gene_names[i]}_{gene_names[j]}"
                    columns[key] = matrix_array[i, j]
        return columns

    # All matrices we want to flatten
    matrix_data = {
        "directional": results["direction_matrix"],
        "gene_gene": results["pairwise_gene_gene_correlation_matrix"],
        "random": results["random_pair_correlation_matrix_t2"],
        "twin_t1": results["twin_pair_correlation_matrix_t1"],
        "twin_t2": results["twin_pair_correlation_matrix_t2"]
    }

    # Flatten matrices and add to output dictionary
    for mtype, matrix in matrix_data.items():
        record.update(matrix_to_gene_pair_columns(matrix, gene_names, f"{mtype}_"))
    return record


def save_results_to_csv(results_list, output_dir="results"):
    """
    Save flattened simulation results to multiple CSV files:
    - all_simulation_results.csv (everything combined)
    - one CSV per matrix type (directional, gene_gene, random, twin_t1, twin_t2)
    - summary CSV aggregated over simulation types

    Parameters
    ----------
    results_list : list[dict]
        List of flattened simulation records created by process_simulation().

    output_dir : str
        Directory where all files will be saved.

    Returns
    -------
    df : pandas.DataFrame
        Full combined DataFrame of all results.
    """

    os.makedirs(output_dir, exist_ok=True)
    matrix_types = ["directional", "gene_gene", "random", "twin_t1", "twin_t2"]

    # Create separate CSV for each matrix type
    for matrix_type in matrix_types:
        matrix_columns = [c for c in df.columns if c.startswith(f"{matrix_type}_")]

        if matrix_columns:
            metadata_cols = ["sim_type", "rep_id", "analysis_key"]
            subset_cols = metadata_cols + matrix_columns
            subset_df = df[subset_cols].copy()

            # Remove prefix for nicer output
            rename_dict = {col: col.replace(f"{matrix_type}_", "") for col in matrix_columns}
            subset_df.rename(columns=rename_dict, inplace=True)

            file_path = os.path.join(output_dir, f"{matrix_type}_matrix_results.csv")
            subset_df.to_csv(file_path, index=False)
            print(f"✅ Saved {matrix_type} matrix to {file_path}")
    return

In [ ]:
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Function to get all files matching a pattern
def get_all_rep_files(pattern):
    return glob.glob(pattern)

def process_all_reps_with_twins(file_pattern_or_folders, base_config, t1, t2, twin_data_df, sim_type_name, is_two_state=False):
    all_medians = []
    z_scores = []
    z_score_rand_list = []
    z_threshold_list = []
    if is_two_state:
        # Handle 2-state cases with paired files
        high_k_files = glob.glob(f"{file_pattern_or_folders[0]}/df_rows_*_rep_*.csv")
        low_k_files = glob.glob(f"{file_pattern_or_folders[1]}/df_rows_*_rep_*.csv")
        
        for high_file in high_k_files:
            # Extract rep number from filename
            rep_num = int(high_file.split('rep_')[-1].split('.csv')[0])
            
            # Find matching low_k file
            matching_low_file = [f for f in low_k_files if f'rep_{rep_num}.csv' in f]
            
            if matching_low_file:
                file_pair = [high_file, matching_low_file[0]]
                print(f"Processing {sim_type_name} rep {rep_num}: {file_pair}")
                
                try:
                    random_correlation = get_many_random_pair_corr(
                        file_pair,
                        base_config,
                        t1, t2,
                        check_for_steady_state=False,
                        have_any_output=False,
                        plot_correlation_matrices_as_heatmap=False
                    )["all_values"]
                    
                    # Calculate median and z-score
                    median_corr = np.median(random_correlation)
                    all_medians.append(median_corr)
                    
                    # Get twin data
                    twin_row = twin_data_df[(twin_data_df['sim_type'] == sim_type_name) & 
                                           (twin_data_df['rep_id'] == rep_num)]
                    random_row = random_correlation_data[(random_correlation_data['sim_type'] == sim_type_name) & 
                                           (random_correlation_data['rep_id'] == rep_num)]
                    if not twin_row.empty:
                        twin_vals = twin_row['g2_g1'].values[0]
                        random_vals = random_row['g2_g1'].values[0]
                        random_mean = np.mean(random_correlation)
                        random_std = np.std(random_correlation)
                        z_score = (twin_vals - random_mean) / random_std
                        z_scores.append(z_score)
                        z_threshold = 10*random_std + random_mean
                        z_threshold_list.append(z_threshold)
                        z_score_rand = (random_vals - random_mean) / random_std
                        z_score_rand_list.append(z_score_rand)
                    else:
                        print(f"Warning: No twin data found for {sim_type_name} rep {rep_num}")
                        z_scores.append(np.nan)
                        
                except Exception as e:
                    print(f"Error processing {file_pair}: {e}")
                    continue
    else:
        # Handle single-state cases
        files = glob.glob(file_pattern_or_folders)
        
        for file_path in files:
            rep_num = int(file_path.split('rep_')[-1].split('.csv')[0])
            print(f"Processing {sim_type_name} rep {rep_num}: {file_path}")
            
            try:
                random_correlation = get_many_random_pair_corr(
                    file_path,
                    base_config,
                    t1, t2,
                    check_for_steady_state=False,
                    have_any_output=False,
                    plot_correlation_matrices_as_heatmap=False
                )["all_values"]
                
                median_corr = np.median(random_correlation)
                all_medians.append(median_corr)
                
                # Get twin data
                twin_row = twin_data_df[(twin_data_df['sim_type'] == sim_type_name) & 
                                    (twin_data_df['rep_id'] == rep_num)]
                random_row = random_correlation_data[(random_correlation_data['sim_type'] == sim_type_name) & 
                                        (random_correlation_data['rep_id'] == rep_num)]
                if not twin_row.empty:
                    twin_vals = twin_row['g2_g1'].values[0]
                    random_vals = random_row['g2_g1'].values[0]
                    random_mean = np.mean(random_correlation)
                    random_std = np.std(random_correlation)
                    z_score = (twin_vals - random_mean) / random_std
                    z_scores.append(z_score)
                    z_threshold = 10*random_std + random_mean
                    z_threshold_list.append(z_threshold)
                    z_score_rand = (random_vals - random_mean) / random_std
                    z_score_rand_list.append(z_score_rand)
                else:
                    print(f"Warning: No twin data found for {sim_type_name} rep {rep_num}")
                    z_scores.append(np.nan)
                    
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                continue
    
    return all_medians, z_scores, z_score_rand_list, z_threshold_list

## Analyze all simulations using TwINFER (run the pipeline using infer_with_twinfer function)

In [ ]:
#Collect all the simulations needed for figure 2: The 4 scenarios.
tasks = []

# Case 1: A_to_B
sim_folder = f"{path_to_simulation_data}/A_to_B/"
pattern = os.path.join(sim_folder, "df_rows_0_1_*_ncells_6000_A_to_B_rep_*.csv")
for f in sorted(glob.glob(pattern)):
    rep_id = os.path.splitext(os.path.basename(f))[0].split("_rep_")[-1]
    tasks.append((f, "A_to_B", rep_id))

# Case 2: A, B no regulation
sim_folder = f"{path_to_simulation_data}/A_B/"
pattern = os.path.join(sim_folder, "df_rows_0_1_*_ncells_6000_A_B_rep*.csv")
for f in sorted(glob.glob(pattern)):
    rep_id = os.path.splitext(os.path.basename(f))[0].split("_rep_")[-1]
    tasks.append((f, "A_B", rep_id))

# Case 3: 2 states A_to_B - generating 2 states by combining simulations with low k_on and high k_on
sim_folder_1 = f"{path_to_simulation_data}/A_to_B_high_k_on/"
sim_folder_2 =  f"{path_to_simulation_data}/A_to_B_low_k_on/"

files_1 = [f for f in glob.glob(os.path.join(sim_folder_1, "*.csv"))
           if os.path.basename(f).startswith('df_')]
files_2 = [f for f in glob.glob(os.path.join(sim_folder_2, "*.csv"))
           if os.path.basename(f).startswith('df_')]

pairs = {}
for f in files_1:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_to_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['high'] = f

for f in files_2:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_to_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['low'] = f

for (sim_type, rep_id), pair in pairs.items():
    if pair['high'] and pair['low']:
        tasks.append(([pair['high'], pair['low']], sim_type, rep_id))

# Case 4: 2 states, A,B - generating 2 states by combining simulations with low k_on and high k_on
sim_folder_1 = f"{path_to_simulation_data}/A_B_high_k_on/"
sim_folder_2 = f"{path_to_simulation_data}/A_B_low_k_on/"

files_1 = [f for f in glob.glob(os.path.join(sim_folder_1, "*.csv"))
           if os.path.basename(f).startswith('df_')]
files_2 = [f for f in glob.glob(os.path.join(sim_folder_2, "*.csv"))
           if os.path.basename(f).startswith('df_')]

pairs = {}
for f in files_1:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['high'] = f

for f in files_2:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['low'] = f

for (sim_type, rep_id), pair in pairs.items():
    if pair['high'] and pair['low']:
        tasks.append(([pair['high'], pair['low']], sim_type, rep_id))


In [ ]:
#Collect all the simulations needed for figure 2: The 4 scenarios.
tasks = []

# Case 1: A_to_B
sim_folder = f"{path_to_simulation_data}/A_to_B/"
pattern = os.path.join(sim_folder, "df_rows_0_1_*_ncells_6000_A_to_B_rep_*.csv")
for f in sorted(glob.glob(pattern)):
    rep_id = os.path.splitext(os.path.basename(f))[0].split("_rep_")[-1]
    tasks.append((f, "A_to_B", rep_id))

# Case 2: A, B no regulation
sim_folder = f"{path_to_simulation_data}/A_B/"
pattern = os.path.join(sim_folder, "df_rows_0_1_*_ncells_6000_A_B_rep*.csv")
for f in sorted(glob.glob(pattern)):
    rep_id = os.path.splitext(os.path.basename(f))[0].split("_rep_")[-1]
    tasks.append((f, "A_B", rep_id))

# Case 3: 2 states A_to_B - generating 2 states by combining simulations with low k_on and high k_on
sim_folder_1 = f"{path_to_simulation_data}/A_to_B_high_k_on/"
sim_folder_2 =  f"{path_to_simulation_data}/A_to_B_low_k_on/"

files_1 = [f for f in glob.glob(os.path.join(sim_folder_1, "*.csv"))
           if os.path.basename(f).startswith('df_')]
files_2 = [f for f in glob.glob(os.path.join(sim_folder_2, "*.csv"))
           if os.path.basename(f).startswith('df_')]

pairs = {}
for f in files_1:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_to_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['high'] = f

for f in files_2:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_to_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['low'] = f

for (sim_type, rep_id), pair in pairs.items():
    if pair['high'] and pair['low']:
        tasks.append(([pair['high'], pair['low']], sim_type, rep_id))

# Case 4: 2 states, A,B - generating 2 states by combining simulations with low k_on and high k_on
sim_folder_1 = f"{path_to_simulation_data}/A_B_high_k_on/"
sim_folder_2 = f"{path_to_simulation_data}/A_B_low_k_on/"

files_1 = [f for f in glob.glob(os.path.join(sim_folder_1, "*.csv"))
           if os.path.basename(f).startswith('df_')]
files_2 = [f for f in glob.glob(os.path.join(sim_folder_2, "*.csv"))
           if os.path.basename(f).startswith('df_')]

pairs = {}
for f in files_1:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['high'] = f

for f in files_2:
    rep_match = re.search(r"rep_(\d+)", os.path.basename(f))
    if rep_match:
        rep_id = rep_match.group(1)
        key = ("A_B_2_states", rep_id)
        pairs.setdefault(key, {'high': None, 'low': None})['low'] = f

for (sim_type, rep_id), pair in pairs.items():
    if pair['high'] and pair['low']:
        tasks.append(([pair['high'], pair['low']], sim_type, rep_id))


In [ ]:
# # ---------- Run all tasks in parallel and save the results in plot_data folder
print("🚀 Starting parallel processing...")

results_list = Parallel(n_jobs=8, backend="loky")(
    delayed(process_simulation_through_infer_with_twinfer)(path, sim_type, rep_id, base_config, t1, t2)
    for path, sim_type, rep_id in tasks
)

# # ---------- Save results to CSV ----------
print("\n💾 Saving results to CSV files...")
df = save_results_to_csv(results_list, output_dir = path_to_save_plot_data)

# ---------- Quick analysis ----------
print(f"\n All simulations processed and saved!")
print(f"Total simulations: {len(df)}")
print(f"Simulation types: {list(df['sim_type'].unique())}")
print(f"Columns created: {len(df.columns)}")

## Test for multiple states using z-score between twin difference correlation and random-pair difference correlation

This is similar to step 2 of TwINFER pipeline - but implemented here separately for clarity and output specific for data for this figure.

In [ ]:
# Load the data from step 1 output
random_correlation_path = f"{path_to_save_plot_data}/random_matrix_results.csv"
twin_correlation_t1_path= f"{path_to_save_plot_data}/twin_t1_matrix_results.csv"
random_correlation_data = pd.read_csv(random_correlation_path)
twin_correlation_t1_data = pd.read_csv(twin_correlation_t1_path)

In [ ]:
# Usage:
# Single state with regulation
A_to_B_medians, A_to_B_z_scores, A_to_B_z_rand_scores, A_to_B_z_threshold_list = process_all_reps_with_twins(
    f"{path_to_simulation_data}/A_to_B/df_rows_*_A_to_B_rep_*.csv", 
    base_config, t1, t2, twin_correlation_t1_data, "A_to_B", is_two_state=False
)

# Two state cases
A_B_2_states_medians, A_B_2_states_z_scores, A_B_2_states_z_rand_scores, A_B_2_states_z_threshold_list = process_all_reps_with_twins(
    [f"{path_to_simulation_data}/A_B_high_k_on", f"{path_to_simulation_data}/A_B_low_k_on"], 
    base_config, t1, t2, twin_correlation_t1_data, "A_B_2_states", is_two_state=True
)

A_to_B_2_states_medians, A_to_B_2_states_z_scores, A_to_B_2_states_z_rand_scores, A_to_B_2_states_z_threshold_list = process_all_reps_with_twins(
    [f"{path_to_simulation_data}/A_to_B_high_k_on", f"{path_to_simulation_data}/A_to_B_low_k_on"], 
    base_config, t1, t2, twin_correlation_t1_data, "A_to_B_2_states", is_two_state=True
)

In [ ]:
#Preparing the output of analysis to save cleanly into csv files for plotting.

# Prepare a dictionary of all lists
data_dict = {
    "network_type": [],
    "metric": [],
    "values": []
}

entries = [
    ("A_to_B", "medians", A_to_B_medians),
    ("A_to_B", "z_scores", A_to_B_z_scores),
    ("A_to_B", "z_rand_scores", A_to_B_z_rand_scores),
    
    ("A_to_B_2_states", "medians", A_to_B_2_states_medians),
    ("A_to_B_2_states", "z_scores", A_to_B_2_states_z_scores),
    ("A_to_B_2_states", "z_rand_scores", A_to_B_2_states_z_rand_scores),

    ("A_B_2_states", "medians", A_B_2_states_medians),
    ("A_B_2_states", "z_scores", A_B_2_states_z_scores),
    ("A_B_2_states", "z_rand_scores", A_B_2_states_z_rand_scores),
]

# Flatten
for network, metric, values in entries:
    for v in values:
        data_dict["network_type"].append(network)
        data_dict["metric"].append(metric)
        data_dict["values"].append(v)

df = pd.DataFrame(data_dict)
output_csv = f"{path_to_save_plot_data}/twins_random_zscore_summary.csv"
df.to_csv(output_csv, index=False)
print("Saved to:", output_csv)
